In [1]:
import torch
import torch.nn as nn

In [2]:
class ConvNextBlock(nn.Module):
    def __init__(self, dim, layer_scale=1e-6):
        super().__init__()
        self.block = nn.Sequential(
            # Depthwise Conv 7×7 (filter lớn hơn ResNet)
            nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim),
            # LayerNorm (thay vì BatchNorm)
            nn.GroupNorm(1, dim),   # GroupNorm(1) ≈ LayerNorm cho ảnh
            # Pointwise mở rộng 4× kênh
            nn.Conv2d(dim, dim * 4, kernel_size=1),
            # GELU (thay vì ReLU)
            nn.GELU(),
            # Pointwise thu hẹp về kích thước cũ
            nn.Conv2d(dim * 4, dim, kernel_size=1),
        )
        # Layer Scale: nhân output với hệ số nhỏ ban đầu
        self.gamma = nn.Parameter(layer_scale * torch.ones(dim, 1, 1))
    def forward(self, x):
        return x + self.gamma * self.block(x)   # Skip Connection

In [3]:
class ConvNextDownsample(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.ds = nn.Sequential(
            nn.GroupNorm(1, in_ch),
            nn.Conv2d(in_ch, out_ch, kernel_size=2, stride=2)  # stride=2 để giảm đôi
        )
    def forward(self, x):
        return self.ds(x)


In [4]:
class ConvNext(nn.Module):
    # depths: số block mỗi stage, dims: số kênh mỗi stage
    def __init__(self, depths=[3,3,9,3], dims=[96,192,384,768], num_classes=1000):
        super().__init__()
        # Stem: Conv 4×4, stride=4 (patchify như ViT)
        self.stem = nn.Sequential(
            nn.Conv2d(3, dims[0], kernel_size=4, stride=4),
            nn.GroupNorm(1, dims[0])
        )
        # 4 Stage với Downsampling ở giữa
        self.stages = nn.ModuleList()
        self.downsamples = nn.ModuleList()
        for i in range(4):
            # Mỗi stage gồm nhiều ConvNextBlock
            stage = nn.Sequential(*[ConvNextBlock(dims[i]) for _ in range(depths[i])])
            self.stages.append(stage)
            # Downsample (trừ stage cuối)
            if i < 3:
                self.downsamples.append(ConvNextDownsample(dims[i], dims[i+1]))
        self.norm = nn.GroupNorm(1, dims[-1])
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(dims[-1], num_classes)
    def forward(self, x):
        x = self.stem(x)                        # 224→56
        for i in range(4):
            x = self.stages[i](x)               # Qua các ConvNextBlock
            if i < 3:
                x = self.downsamples[i](x)      # 56→28→14→7
        x = self.norm(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


In [5]:
model = ConvNext(depths=[3,3,9,3], dims=[96,192,384,768], num_classes=1000)
dummy = torch.randn(2, 3, 224, 224)
out = model(dummy)
print(f"Output shape: {out.shape}")    # (2, 1000)
total = sum(p.numel() for p in model.parameters())
print(f"Total params: {total:,}")      # ~28M (Tiny)

Output shape: torch.Size([2, 1000])
Total params: 28,589,128
